In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込みエラー: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 23. KV Cache

> **学習目標**
> - KV Cache 推論 速度 $O(n^2) \to O(n)$
> - Prefill vs Decode
> - 計算
> - KV Cache / 速度 比較

## 23.1 推論 : Prefill vs Decode

LLM 推論 :
1. **Prefill (Context Encoding)**: . . $O(n^2)$.
2. **Decode (Token Generation)**: . . $O(n)$.

Decode 時間 . KV Cache decode 速度 .

## 23.2 K, V

Attention: $\mathrm{softmax}(QK^\top/\sqrt{d_k})V$

 $t$ :
- $Q_t$ ( )
- $K_{\leq t}, V_{\leq t}$ ( , 値)

 $K, V$ 計算 → $O(t^2)$.
 $K_{\leq t-1}, V_{\leq t-1}$ $K_t, V_t$ → $O(t)$.

**Q **: $Q_t$ .


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import time

#   Attention (   計算)
def attention_no_cache(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-1, -2) / (d_k ** 0.5)
    if mask is not None:
        scores = scores + mask
    weights = F.softmax(scores, dim=-1)
    return weights @ V

#   Attention ()
def attention_with_cache(q_new, k_cache, v_cache, k_new, v_new):
    """q_new: (B, 1, d), k_cache/v_cache: (B, T, d)"""
    # Cache  k, v 
    k_full = torch.cat([k_cache, k_new], dim=1)  # (B, T+1, d)
    v_full = torch.cat([v_cache, v_new], dim=1)
    d_k = q_new.shape[-1]
    scores = q_new @ k_full.transpose(-1, -2) / (d_k ** 0.5)  # (B, 1, T+1)
    weights = F.softmax(scores, dim=-1)
    out = weights @ v_full  # (B, 1, d)
    return out, k_full, v_full

# Comparison: 100  Generation
np.random.seed(0)
torch.manual_seed(0)
B, d, T = 1, 64, 1
n_generate = 50

# Cache  
t0 = time.perf_counter()
Q = torch.randn(B, T, d)
K = torch.randn(B, T, d)
V = torch.randn(B, T, d)
for step in range(n_generate):
    #  Step  Computation
    Q_new = torch.randn(B, 1, d)
    Q = torch.cat([Q, Q_new], dim=1)
    K = torch.cat([K, torch.randn(B, 1, d)], dim=1)
    V = torch.cat([V, torch.randn(B, 1, d)], dim=1)
    out = attention_no_cache(Q, K, V)
t_no_cache = time.perf_counter() - t0
print(f" : {t_no_cache*1000:.2f}ms ({n_generate} )")

# Cache  
torch.manual_seed(0)
t0 = time.perf_counter()
k_cache = torch.randn(B, 1, d)
v_cache = torch.randn(B, 1, d)
for step in range(n_generate):
    q_new = torch.randn(B, 1, d)
    k_new = torch.randn(B, 1, d)
    v_new = torch.randn(B, 1, d)
    out, k_cache, v_cache = attention_with_cache(q_new, k_cache, v_cache, k_new, v_new)
t_cache = time.perf_counter() - t0
print(f" : {t_cache*1000:.2f}ms ({n_generate} )")
print(f"Speed Improvement: {t_no_cache/t_cache:.2f}x")


## 23.3 KV Cache 計算

KV Cache = $2 \cdot L \cdot h \cdot d_k \cdot n \cdot \text{dtype}$

- $L$:
- $h$: KV (MHA $h$, MQA 1, GQA $g$)
- $d_k$: 次元
- $n$: シーケンス長
- 2: K V
- dtype: FP16=2 bytes, FP32=4 bytes

: LLaMA-7B ($L=32, h=32, d_k=128$), FP16, $n=4096$:
$$2 \times 32 \times 32 \times 128 \times 4096 \times 2 = 2 \text{ GB}$$

$n=32768$ 16GB → GPU .


In [ ]:
# KV Cache  計算
def kv_cache_memory_gb(n_layers, n_kv_heads, d_k, seq_len, dtype_bytes=2):
    """KV Cache Memory (GB)."""
    return 2 * n_layers * n_kv_heads * d_k * seq_len * dtype_bytes / (1024**3)

#  Model/Sequence Length
models = [
    ('LLaMA-7B', 32, 32, 128, 2),
    ('LLaMA-13B', 40, 40, 128, 2),
    ('LLaMA-70B', 80, 8, 128, 2),  # GQA
    ('GPT-2 small', 12, 12, 64, 2),
    ('Mistral-7B', 32, 8, 128, 2),  # GQA
]
seq_lens = [2048, 8192, 32768, 131072]

print(f"{'Model':>15} | ", end='')
for sl in seq_lens:
    print(f"n={sl:>6}", end=' | ')
print()
print("-" * 75)
for name, L, h, dk, dt in models:
    print(f"{name:>15} | ", end='')
    for sl in seq_lens:
        mem = kv_cache_memory_gb(L, h, dk, sl, dt)
        print(f"{mem:>7.2f}GB", end=' | ')
    print()

print("\n=> Sequence Length の場合 KV Cache Memory Subspace .")
print("=> GQA(Mistral, LLaMA-70B) KV Head   Memory Efficiency.")


## 23.4 [CPU/GPU ベンチマーク ⑩] KV Cache on/off


In [ ]:
# KV Cache ベンチマーク
from llm_math.bench import time_fn

#  Attention    
def bench_decode_no_cache(n_context, d=64, n_decode=20):
    """Cache  n_decode  Generation."""
    Q = torch.randn(1, n_context, d)
    K = torch.randn(1, n_context, d)
    V = torch.randn(1, n_context, d)
    for _ in range(n_decode):
        Q_new = torch.randn(1, 1, d)
        Q = torch.cat([Q, Q_new], dim=1)
        K = torch.cat([K, torch.randn(1, 1, d)], dim=1)
        V = torch.cat([V, torch.randn(1, 1, d)], dim=1)
        _ = Q @ K.transpose(-1, -2) / (d ** 0.5) @ V

def bench_decode_with_cache(n_context, d=64, n_decode=20):
    """Cache  n_decode  Generation."""
    k_cache = torch.randn(1, n_context, d)
    v_cache = torch.randn(1, n_context, d)
    for _ in range(n_decode):
        q_new = torch.randn(1, 1, d)
        k_new = torch.randn(1, 1, d)
        v_new = torch.randn(1, 1, d)
        k_full = torch.cat([k_cache, k_new], dim=1)
        v_full = torch.cat([v_cache, v_new], dim=1)
        _ = q_new @ k_full.transpose(-1, -2) / (d ** 0.5) @ v_full
        k_cache = k_full
        v_cache = v_full

print(f"{'n_context':>10} | {'No Cache (ms)':>15} | {'Cache (ms)':>12} | {'Speedup':>10}")
print("-" * 55)
for n_ctx in [128, 512, 1024, 2048]:
    t_no = time_fn(bench_decode_no_cache, n_ctx, device='cpu', warmup=1, repeat=3)['mean_ms']
    t_yes = time_fn(bench_decode_with_cache, n_ctx, device='cpu', warmup=1, repeat=3)['mean_ms']
    print(f"{n_ctx:>10} | {t_no:>15.3f} | {t_yes:>12.3f} | {t_no/t_yes:>9.2f}x")
print("\n=> Context  KV Cache Effect  (O(n) vs O(n^2)).")


## 23.5 Continuous Batching

 batching: → .

**Continuous Batching** (vLLM ): .
- (thoughtput)
-

## 23.6 PagedAttention (vLLM)

OS :
- KV Cache ""
-
-

 , . vLLM 2-4x .

## 23.7 要点

| | |
|---|---|
| Prefill | , $O(n^2)$ |
| Decode | , $O(n)$ per token (KV cache) |
| KV Cache | $K, V$ , $K_t, V_t$ |
| | $2 L h d_k n \cdot \text{dtype}$ |
| Continuous Batching | , |
| PagedAttention | |

## 演習問題

1. KV Cache on/off 100 時間 n=128, 512, 2048 比較.
2. LLaMA-7B 32K KV Cache 計算.
3. GQA KV Cache LLaMA-7B(32 ) vs LLaMA-70B(8 KV ) 比較.
4. Continuous Batching .
5. PagedAttention .

> 解答: `solutions/ch23_solutions.ipynb`
